# 04 · Final model, testing and business interpretation (Team 10, Saideep)

**Deliverable 5.** Take the baseline chosen in D4 (**Ridge regression on log price, all D3
features**; see `results/baseline-decision.md`), finalise its training, test it once on held-out
data, describe how it behaves, and say what it means for American Airlines.

### About the two-year forecast horizon

The D5 brief asks for a demand forecast two years ahead. **This dataset cannot support that, and
we are not going to fake it.**

- The target is a **fare** (`total_price` of one quote), not **demand** (passengers).
- There is **no date column**: no search date, no departure date, no year. Nothing orders the
  rows in time, so there is no "past" to train on and no "future" to test on, at any horizon.
- `month` is the departure month-of-year, not a calendar date, so it gives seasonality but no trend.

The most honest result this file supports is a **cross-sectional pricing model**: given an
itinerary, cabin, booking window, load and competitor quote *at quote time*, what fare does AA
charge? Its "horizon" is the booking window (0–119 days before departure), not years. What a
two-year demand forecast would need is listed at the end of this notebook and in
`results/forecast-evaluation.md`.

### Test design

- **Held-out test set:** 20% of cleaned rows (40 quotes), stratified by cabin, `random_state=10`.
  It is touched exactly once, at the end. A random split is the only option without dates.
- **All tuning happens on the 159 training rows** with repeated 5-fold CV.
- **Baselines on the same test rows:** cabin median, match-the-competitor rule, and the D4
  fallback (XGBoost without the competitor price).

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error,
                             mean_squared_error, r2_score)
from sklearn.model_selection import (GridSearchCV, RepeatedKFold, cross_val_predict,
                                     learning_curve, train_test_split, KFold)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from xgboost import XGBRegressor

HERE = next(p for p in [Path.cwd(), Path.cwd() / "team-10-modeling" / "notebooks"]
            if (p / "load_data.py").exists())
sys.path.insert(0, str(HERE))
MODELING = HERE.parent
RESULTS, VISUALS, MODELS = MODELING / "results", MODELING / "visuals", MODELING / "models"
for d in (RESULTS, VISUALS, MODELS):
    d.mkdir(exist_ok=True)

from load_data import CABIN_ORDER, load_data
from features import FEATURES, FEATURES_NO_COMPETITOR, build_features, clean

SEED = 10
sns.set_theme(style="whitegrid")
df = clean(load_data())
X, y = build_features(df)

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X, y, df, test_size=0.2, stratify=df["cabin"], random_state=SEED)
print("train:", X_train.shape, "| test:", X_test.shape)
print(df_test["cabin"].value_counts().to_string())

## 1. Finalise training: tune the Ridge pipeline on the training rows only

Search space:
- `alpha` (regularisation strength), 1e-3 to 1e3.
- Whether to add **pairwise interaction terms** (e.g. cabin × last-minute). D3 suspected
  interactions; this tests it. Pairwise terms take 16 features to 152, so regularisation matters.

Scaling sits inside the pipeline, so each CV fold fits its own scaler (no leakage from the
validation fold).

In [ ]:
def ridge_pipeline():
    pipe = Pipeline([
        ("interactions", "passthrough"),
        ("scale", StandardScaler()),
        ("ridge", Ridge()),
    ])
    return TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)

grid = {
    "regressor__interactions": ["passthrough",
                                PolynomialFeatures(degree=2, interaction_only=True,
                                                   include_bias=False)],
    "regressor__ridge__alpha": np.logspace(-3, 3, 25),
}
inner_cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=SEED)
search = GridSearchCV(ridge_pipeline(), grid, cv=inner_cv,
                      scoring="neg_mean_absolute_error")
search.fit(X_train, y_train)

cvres = pd.DataFrame(search.cv_results_)
cvres["interactions"] = cvres["param_regressor__interactions"].map(
    lambda v: "none" if v == "passthrough" else "pairwise")
cvres["alpha"] = cvres["param_regressor__ridge__alpha"].astype(float)
cvres["cv_MAE"] = -cvres["mean_test_score"]
best = search.best_params_
best_inter = "none" if best["regressor__interactions"] == "passthrough" else "pairwise"
best_alpha = float(best["regressor__ridge__alpha"])
print(f"best: interactions={best_inter}, alpha={best_alpha:.4g}, CV MAE=${-search.best_score_:.2f}")
print(cvres.groupby("interactions")["cv_MAE"].min().round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 4.5))
for name, g in cvres.groupby("interactions"):
    ax.plot(g["alpha"], g["cv_MAE"], marker="o", ms=3, label=f"interactions: {name}")
ax.axvline(best_alpha, color="k", ls="--", lw=1)
ax.set(xscale="log", xlabel="alpha (regularisation strength)", ylabel="CV MAE (USD)",
       title="Ridge tuning on the training rows")
ax.legend()
fig.tight_layout()
fig.savefig(VISUALS / "d5_ridge_tuning.png", dpi=120)
plt.show()

final_model = search.best_estimator_

## 2. Learning curve: data problem or model problem?

In [ ]:
sizes, tr_scores, va_scores = learning_curve(
    clone(final_model), X_train, y_train, cv=KFold(5, shuffle=True, random_state=SEED),
    train_sizes=np.linspace(0.2, 1.0, 8), scoring="neg_mean_absolute_error")
lc = pd.DataFrame({"n_train": sizes, "train_MAE": -tr_scores.mean(1), "val_MAE": -va_scores.mean(1)})
print(lc.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(lc["n_train"], lc["train_MAE"], marker="o", label="training MAE")
ax.plot(lc["n_train"], lc["val_MAE"], marker="o", label="validation MAE")
ax.set(xlabel="training rows", ylabel="MAE (USD)", title="Learning curve: final Ridge model")
ax.legend()
fig.tight_layout()
fig.savefig(VISUALS / "d5_learning_curve.png", dpi=120)
plt.show()

## 3. Prediction intervals (split-conformal on log residuals)

Ridge gives a point estimate. To give a range, we take **out-of-fold** residuals on the
training rows (log scale) and use their empirical quantiles as the interval half-widths. Because
the residuals are on the log scale, the dollar interval scales with the fare: wider for a
$1,000 Business fare than for a $90 shuttle fare. Coverage is then checked on the untouched test set.

In [ ]:
oof = cross_val_predict(clone(final_model), X_train, y_train,
                        cv=KFold(5, shuffle=True, random_state=SEED))
log_resid = np.log1p(y_train.to_numpy()) - np.log1p(oof)
q = {lvl: np.quantile(log_resid, [(1 - lvl) / 2, 1 - (1 - lvl) / 2]) for lvl in (0.8, 0.9)}
for lvl, (lo, hi) in q.items():
    print(f"{int(lvl * 100)}% interval: price × [{np.exp(lo):.3f}, {np.exp(hi):.3f}]")


def interval(pred, lvl):
    lo, hi = q[lvl]
    return np.expm1(np.log1p(pred) + lo), np.expm1(np.log1p(pred) + hi)

## 4. Test once on the held-out rows

In [ ]:
class CabinMedian:
    def fit(self, X, y):
        self.m = pd.Series(y.to_numpy()).groupby(X["cabin_rank"].to_numpy()).median(); return self
    def predict(self, X):
        return X["cabin_rank"].map(self.m).to_numpy()


class MatchCompetitor:
    def fit(self, X, y):
        self.r = float(np.median(y.to_numpy() / np.exp(X["log_competitor_price"].to_numpy()))); return self
    def predict(self, X):
        return self.r * np.exp(X["log_competitor_price"].to_numpy())


xgb_fallback = TransformedTargetRegressor(
    regressor=XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=3, subsample=0.8,
                           colsample_bytree=0.8, min_child_weight=2, random_state=SEED, n_jobs=2),
    func=np.log1p, inverse_func=np.expm1)

final_model.fit(X_train, y_train)
preds = {
    "Final Ridge (log price)": final_model.predict(X_test),
    "Match-the-competitor rule": MatchCompetitor().fit(X_train, y_train).predict(X_test),
    "Cabin median": CabinMedian().fit(X_train, y_train).predict(X_test),
    "XGBoost, no competitor (fallback)": xgb_fallback.fit(
        X_train[FEATURES_NO_COMPETITOR], y_train).predict(X_test[FEATURES_NO_COMPETITOR]),
}


def scores(a, p):
    return {"MAE": mean_absolute_error(a, p), "RMSE": np.sqrt(mean_squared_error(a, p)),
            "R2": r2_score(a, p), "MAPE_%": 100 * mean_absolute_percentage_error(a, p),
            "max_abs_err": np.max(np.abs(a - p))}

test_scores = pd.DataFrame({k: scores(y_test, v) for k, v in preds.items()}).T
p_final = preds["Final Ridge (log price)"]
coverage = {}
for lvl in (0.8, 0.9):
    lo, hi = interval(p_final, lvl)
    coverage[lvl] = float(((y_test >= lo) & (y_test <= hi)).mean())
    print(f"{int(lvl * 100)}% interval coverage on test: {coverage[lvl]:.1%} "
          f"(median width ${np.median(hi - lo):.0f})")
test_scores.round(3)

## 5. What the model does: diagnostics

In [ ]:
lo80, hi80 = interval(p_final, 0.8)
resid = y_test.to_numpy() - p_final
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ax.errorbar(y_test, p_final, yerr=[p_final - lo80, hi80 - p_final], fmt="o", ms=4,
            alpha=0.7, elinewidth=0.8, label="prediction ± 80% interval")
lim = [25, max(y_test.max(), p_final.max()) * 1.1]
ax.plot(lim, lim, "k--", lw=1, label="perfect")
ax.set(xscale="log", yscale="log", xlabel="observed total_price (USD)",
       ylabel="predicted (USD)", title="Held-out test: observed vs predicted")
ax.legend()
ax = axes[1]
ax.scatter(p_final, resid / y_test.to_numpy() * 100, s=18)
ax.axhline(0, color="k", lw=1)
ax.set(xscale="log", xlabel="predicted (USD)", ylabel="error (% of observed)",
       title="Held-out test: relative residuals")
fig.tight_layout()
fig.savefig(VISUALS / "d5_observed_vs_predicted.png", dpi=120)
plt.show()

### Coefficients

The pipeline standardises features, so each coefficient is the change in **log price per one
standard deviation** of the feature, holding the others fixed. `exp(coef)` is the price
multiplier. If the search picked pairwise interactions, the table lists the largest terms.

In [ ]:
reg = final_model.regressor_
inter_step = reg.named_steps["interactions"]
names = (list(X.columns) if inter_step == "passthrough"
         else list(inter_step.get_feature_names_out(X.columns)))
coef = pd.Series(reg.named_steps["ridge"].coef_, index=names)
coef_table = (pd.DataFrame({"coef_log": coef, "multiplier_per_sd": np.exp(coef)})
              .reindex(coef.abs().sort_values(ascending=False).index))
print(coef_table.head(15).round(3).to_string())

top = coef_table.head(12)
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top.index[::-1], top["coef_log"][::-1])
ax.axvline(0, color="k", lw=1)
ax.set(xlabel="coefficient (log price per 1 SD)", title="Final Ridge: largest coefficients")
fig.tight_layout()
fig.savefig(VISUALS / "d5_coefficients.png", dpi=120)
plt.show()

### Permutation importance on the test rows

This is model-agnostic: how much test MAE gets worse when one input is shuffled.

In [ ]:
perm = permutation_importance(final_model, X_test, y_test, scoring="neg_mean_absolute_error",
                              n_repeats=30, random_state=SEED)
imp = (pd.DataFrame({"MAE_increase": perm.importances_mean, "std": perm.importances_std},
                    index=X.columns).sort_values("MAE_increase", ascending=False))
imp.round(2)

### Segment-level errors

Where does the model miss? Cross-validated (out-of-fold) predictions on **all** 199 rows are
used here, not just the 40 test rows, because 40 rows split four ways leaves single-digit
segments.

In [ ]:
oof_all = cross_val_predict(clone(final_model), X, y,
                            cv=KFold(5, shuffle=True, random_state=SEED))
seg = df.assign(pred=oof_all, abs_err=np.abs(y - oof_all), pct_err=np.abs(y - oof_all) / y * 100,
                window=np.where(df["days_to_departure"] <= 14, "0-14 days", "15+ days"),
                haul=np.where(df["distance_miles"] > 2200, "long-haul (>2200 mi)", "other"))


def seg_table(col):
    return (seg.groupby(col, observed=True)
               .agg(n=("abs_err", "size"), MAE=("abs_err", "mean"), MAPE_pct=("pct_err", "mean"))
               .round(1))

segments = {c: seg_table(c) for c in ["cabin", "window", "nonstop", "haul"]}
for c, t in segments.items():
    print(f"\n--- by {c} ---\n{t.to_string()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.boxplot(data=seg, x="cabin", y="pct_err", order=CABIN_ORDER, ax=axes[0])
axes[0].set(ylabel="abs error (% of price)", title="Out-of-fold error by cabin")
axes[0].tick_params(axis="x", rotation=15)
sns.boxplot(data=seg, x="window", y="pct_err", ax=axes[1])
axes[1].set(ylabel="abs error (% of price)", title="Out-of-fold error by booking window")
fig.tight_layout()
fig.savefig(VISUALS / "d5_segment_errors.png", dpi=120)
plt.show()

## 6. Save the final model and the generated tables

The model is refit on **all** cleaned rows before saving (the test set's job is done).
`results/test_scores.md` is regenerated on every run; `final-model-report.md` and
`forecast-evaluation.md` interpret it.

In [ ]:
deploy_model = clone(final_model).fit(X, y)
joblib.dump({"model": deploy_model, "features": FEATURES, "log_interval_quantiles": q},
            MODELS / "final_ridge.joblib")
print("saved", MODELS / "final_ridge.joblib")


def md_table(frame, index_name, digits=2):
    cols = list(frame.columns)
    out = [f"| {index_name} | " + " | ".join(cols) + " |", "|" + "---|" * (len(cols) + 1)]
    for idx in frame.index:
        cells = []
        for c in cols:
            v = frame.at[idx, c]
            is_int = pd.api.types.is_integer_dtype(frame[c])
            cells.append(str(v) if is_int or not isinstance(v, (float, np.floating))
                         else f"{v:.{digits}f}")
        out.append(f"| {idx} | " + " | ".join(cells) + " |")
    return "\n".join(out)

text = f"""# Test scores — Team 10 (Deliverable 5)

_Generated by `team-10-modeling/notebooks/04_final_model_saideep.ipynb`. Do not hand-edit; re-run the notebook._

- Train / test: {len(X_train)} / {len(X_test)} rows (random, stratified by cabin, seed {SEED}; no time column exists)
- Final model: Ridge on log1p(price), interactions = **{best_inter}**, alpha = **{best_alpha:.4g}**
- Training-set CV MAE of the chosen settings: ${-search.best_score_:.2f}

## Held-out test (40 rows, touched once)

{md_table(test_scores, "model")}

## Prediction-interval coverage (final model, test rows)

| nominal | observed coverage | price multiplier range |
|---|---|---|
| 80% | {coverage[0.8]:.1%} | × [{np.exp(q[0.8][0]):.3f}, {np.exp(q[0.8][1]):.3f}] |
| 90% | {coverage[0.9]:.1%} | × [{np.exp(q[0.9][0]):.3f}, {np.exp(q[0.9][1]):.3f}] |

## Learning curve (5-fold CV on training rows)

{md_table(lc.set_index("n_train"), "n_train")}

## Largest coefficients (log price per 1 SD)

{md_table(coef_table.head(12), "feature", 3)}

## Permutation importance (test MAE increase, USD)

{md_table(imp, "feature")}

## Segment errors (out-of-fold, all {len(X)} rows)

""" + "\n\n".join(f"### By {c}\n\n{md_table(t, c, 1)}" for c, t in segments.items()) + "\n"
(RESULTS / "test_scores.md").write_text(text, encoding="utf-8")
print("wrote", RESULTS / "test_scores.md")

## 7. Business interpretation and what a two-year forecast would need

The written interpretation is in `results/final-model-report.md`. In short:

- **What this supports:** a transparent fare-benchmark tool. For any quote, it gives the price
  AA's own pattern implies (from the competitor quote, cabin, booking window, distance and load)
  with an interval. Quotes far outside the interval are worth a human look: a filing error like
  the $3,804 row removed in D3, or a pricing opportunity.
- **What it does not support:** demand or capacity planning, or anything "two years out".
  There are no dates and no passenger counts.
- **To get to a two-year demand forecast**, the data needs: (1) dated observations over
  several years (search and departure dates), (2) a demand target (bookings or passengers per
  route-month), and (3) at least two to three years of history, so a full two-year hold-out can
  be tested with rolling origins.